# V_4 · Thời gian, thời tiết và lịch sử lưu lượng

**Mục tiêu:** đo giá trị bổ sung của lịch sử (đối chứng B–C) và đánh giá baseline seasonal naive, baseline còn thiếu trong ba baseline của proposal.

**Kịch bản:** dự đoán tuần tự từng giờ. Tại giờ `t`, các quan sát trước `t` đã được ghi nhận, kể cả các giờ validation trước đó. Đây không phải dự đoán đồng thời toàn bộ tương lai.

| Đặc trưng | Cách tính |
|---|---|
| `lag_24h` | Lưu lượng tại `t − 24 giờ`, tra theo timestamp |
| `lag_168h` | Lưu lượng tại `t − 168 giờ`, tra theo timestamp |
| `mean_24h` | Trung bình các giờ quan sát trong `[t − 24 giờ, t)`, cần ít nhất 18 giờ |

**Chạy trước:** `baseline_V_3.ipynb`. Test giữ riêng.

## 4.1 · Chuẩn bị dữ liệu

In [1]:
from pathlib import Path
import json
import os
import platform
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import xgboost

# Tìm gốc dự án từ thư mục hiện tại hoặc các thư mục cha.
current_dir = Path.cwd().resolve()
project_dir = next(
    (p for p in (current_dir, *current_dir.parents)
     if (p / "data" / "Metro_Interstate_Traffic_Volume.csv").is_file()),
    None,
)
if project_dir is None:
    raise FileNotFoundError("Không tìm thấy CSV trong thư mục dự án.")
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

output_dir = project_dir / "outputs"
figure_dir = project_dir / "figures"
pd.set_option("display.max_columns", None)
plt.rcParams.update({
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#e5e4e0", "axes.axisbelow": True,
    "lines.linewidth": 2,
})
BLUE, ORANGE = "#2a78d6", "#eb6834"

print("Python:", platform.python_version())
print("pandas:", pd.__version__, "| scikit-learn:", sklearn.__version__, "| xgboost:", xgboost.__version__)

Python: 3.13.9
pandas: 3.0.6 | scikit-learn: 1.9.1 | xgboost: 3.4.1


In [2]:
from src.data import DATA_PATH as data_path
from src.data import load_splits
from src.evaluation import (
    bootstrap_mae_difference, regression_metrics, run_observations,
    seasonal_naive, weekday_hour_baseline,
)
from src.experiment_logger import save_experiment_log
from src.features import CONFIG_B, CONFIG_C, HISTORY_FEATURES, build_features, has_history
from src.models import load_predictions, run_config, save_results, summarize_search

In [3]:
# Cùng quy tắc với notebooks/preprocessing_features.ipynb và cùng mốc chia với V_1.
splits = load_splits()
features = build_features(splits)
train, validation = features["train"], features["validation"]

assert (len(train), len(validation), len(features["test"])) == (28402, 6086, 6087)
assert train["date_time"].max() < validation["date_time"].min()
display(pd.DataFrame(
    {"so_gio": len(part), "tu_gio": part["date_time"].min(), "den_gio": part["date_time"].max()}
    for part in features.values()
).set_axis(features.keys()))

,so_gio,tu_gio,den_gio
train,28402,2012-10-02 09:00:00,2017-05-10 04:00:00
validation,6086,2017-05-10 05:00:00,2018-01-19 14:00:00
test,6087,2018-01-19 15:00:00,2018-09-30 23:00:00


## 4.2 · Độ bao phủ của lịch sử

Dữ liệu thiếu 11.976 giờ nên một số giờ không có đủ lịch sử. Giờ thiếu lịch sử giữ `NaN`, không nội suy mục tiêu.

In [4]:
coverage = pd.DataFrame({
    name: {**part[HISTORY_FEATURES].notna().mean().to_dict(),
           "du_ca_ba": has_history(part).mean(), "so_gio_du": int(has_history(part).sum())}
    for name, part in [("train", train), ("validation", validation)]
}).T
display(coverage.round(4))

# Kiểm tra không dùng dữ liệu tại giờ dự đoán hoặc tương lai.
volume = pd.concat([train, validation]).set_index("date_time")["traffic_volume"]
sample = validation[has_history(validation)].sample(300, random_state=42)
for row in sample.itertuples():
    window = volume[(volume.index >= row.date_time - pd.Timedelta(hours=24)) & (volume.index < row.date_time)]
    assert volume[row.date_time - pd.Timedelta(hours=24)] == row.lag_24h
    assert volume[row.date_time - pd.Timedelta(hours=168)] == row.lag_168h
    assert abs(window.mean() - row.mean_24h) < 1e-6
print("Đạt kiểm tra thời điểm của đặc trưng lịch sử trên 300 giờ ngẫu nhiên.")

train_eligible = train[has_history(train)].reset_index(drop=True)
validation_eligible_mask = has_history(validation).to_numpy()
validation_eligible = validation[validation_eligible_mask].reset_index(drop=True)
print("Train đủ lịch sử:", len(train_eligible), "/", len(train))
print("Validation đủ lịch sử:", len(validation_eligible), "/", len(validation))

,lag_24h,lag_168h,mean_24h,du_ca_ba,so_gio_du
train,0.9247,0.8937,0.8771,0.7923,22503.0
validation,0.9969,0.9970,1.0000,0.9941,6050.0


Đạt kiểm tra thời điểm của đặc trưng lịch sử trên 300 giờ ngẫu nhiên.
Train đủ lịch sử: 22503 / 28402
Validation đủ lịch sử: 6050 / 6086


**Nhận xét**

- Validation có 6.050/6.086 giờ (99,41%) đủ cả ba đặc trưng lịch sử vì ít giờ thiếu. Train chỉ có 22.503/28.402 giờ (79,23%) do các đoạn thiếu lớn năm 2013–2015.
- Kiểm tra trên 300 giờ ngẫu nhiên xác nhận `lag_24h`, `lag_168h` và `mean_24h` lấy đúng giờ trong quá khứ, không gồm giờ đang dự đoán.
- Mọi so sánh trong V_4 dùng cùng 6.050 giờ validation.

## 4.3 · Huấn luyện và đối chứng B–C

Mọi phương án được đo trên **cùng các giờ validation đủ lịch sử**. Để không quy phần chênh lệch do số mẫu train cho đặc trưng mới, `B_du_lich_su` và `C_lich_su` dùng cùng tập train đủ lịch sử; `B_toan_bo_train` lấy từ V_3 để thấy ảnh hưởng của việc giảm dữ liệu train.

In [5]:
results = (
    run_config("B_du_lich_su", CONFIG_B, train_eligible, validation_eligible)
    + run_config("C_lich_su", CONFIG_C, train_eligible, validation_eligible)
    # Giữ cả các giờ train thiếu lịch sử: cây nhận NaN, Ridge điền median kèm cờ thiếu trong fold.
    + run_config("C_toan_bo_train", CONFIG_C, train, validation_eligible)
)
save_results("V_4", results, validation_eligible, output_dir)

previous = load_predictions("V_3", output_dir)
assert previous["date_time"].equals(validation["date_time"])
y_eligible = validation_eligible["traffic_volume"]

rows = [
    {"config": "Trung bình thứ–giờ", "model": "Baseline",
     **regression_metrics(y_eligible, weekday_hour_baseline(train, validation_eligible))},
    {"config": "Seasonal naive t − 7 ngày", "model": "Baseline",
     **regression_metrics(y_eligible, seasonal_naive(validation_eligible))},
]
for model in ["Ridge", "Random Forest", "XGBoost"]:
    rows.append({"config": "B_toan_bo_train", "model": model, **regression_metrics(
        y_eligible, previous[f"B_thoi_tiet|{model}"].to_numpy()[validation_eligible_mask])})
rows += [{"config": item["config"], "model": item["model"], **item["metrics"]} for item in results]
summary = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
summary.to_csv(output_dir / "metrics" / "V_4_so_sanh_cung_gio.csv", index=False)
print("Số giờ đánh giá:", len(validation_eligible))
display(summary.round(3))

Số giờ đánh giá: 6050


,config,model,MAE,RMSE,R2
0,C_toan_bo_train,XGBoost,192.940,311.106,0.975
1,C_toan_bo_train,Random Forest,193.203,318.327,0.974
2,C_lich_su,XGBoost,196.132,313.724,0.974
3,C_lich_su,Random Forest,196.775,326.773,0.972
4,B_toan_bo_train,XGBoost,243.041,387.687,0.961
5,C_lich_su,Ridge,243.889,405.786,0.957
6,B_toan_bo_train,Random Forest,245.402,397.185,0.959
7,C_toan_bo_train,Ridge,248.096,411.983,0.956
8,B_du_lich_su,XGBoost,254.495,397.758,0.959
9,B_du_lich_su,Random Forest,258.943,416.099,0.955


**Nhận xét**

- Seasonal naive (lưu lượng cùng giờ tuần trước) có MAE 330,18, kém hơn baseline trung bình thứ–giờ (284,98): một quan sát đơn lẻ chứa nhiễu và ngày lễ của tuần trước, trong khi trung bình thứ–giờ làm mượt qua nhiều tuần.
- Thêm lịch sử cho kết quả tốt nhất: `C_toan_bo_train` với XGBoost đạt MAE 192,94, RMSE 311,11, R² 0,975; Random Forest đạt 193,20.
- Giữ các giờ train thiếu lịch sử (`C_toan_bo_train`) tốt hơn chỉ dùng giờ đủ lịch sử (`C_lich_su`, 196,13) với mô hình cây. Ridge thì ngược lại (248,10 so với 243,89), có thể do cách điền median cho lịch sử thiếu.

In [6]:
by_key = {(item["config"], item["model"]): item for item in results}
rows = []
for model in ["Ridge", "Random Forest", "XGBoost"]:
    rows.append({"so_sanh": "B_du_lich_su → C_lich_su", "model": model, **bootstrap_mae_difference(
        validation_eligible, by_key[("B_du_lich_su", model)]["prediction"], by_key[("C_lich_su", model)]["prediction"])})
    rows.append({"so_sanh": "B_toan_bo_train → B_du_lich_su", "model": model, **bootstrap_mae_difference(
        validation_eligible, previous[f"B_thoi_tiet|{model}"].to_numpy()[validation_eligible_mask],
        by_key[("B_du_lich_su", model)]["prediction"])})
    rows.append({"so_sanh": "B_toan_bo_train → C_toan_bo_train", "model": model, **bootstrap_mae_difference(
        validation_eligible, previous[f"B_thoi_tiet|{model}"].to_numpy()[validation_eligible_mask],
        by_key[("C_toan_bo_train", model)]["prediction"])})
rows.append({"so_sanh": "Seasonal naive → C_lich_su", "model": "XGBoost", **bootstrap_mae_difference(
    validation_eligible, seasonal_naive(validation_eligible), by_key[("C_lich_su", "XGBoost")]["prediction"])})
rows.append({"so_sanh": "Trung bình thứ–giờ → C_lich_su", "model": "XGBoost", **bootstrap_mae_difference(
    validation_eligible, weekday_hour_baseline(train, validation_eligible), by_key[("C_lich_su", "XGBoost")]["prediction"])})
display(pd.DataFrame(rows).round(2))

,so_sanh,model,chenh_lech_MAE,CI95_thap,CI95_cao,so_ngay
0,B_du_lich_su → C_lich_su,Ridge,47.09,36.16,57.59,255
1,B_toan_bo_train → B_du_lich_su,Ridge,-10.55,-12.45,-8.60,255
2,B_toan_bo_train → C_toan_bo_train,Ridge,32.34,24.87,39.83,255
3,B_du_lich_su → C_lich_su,Random Forest,62.17,50.01,75.76,255
4,B_toan_bo_train → B_du_lich_su,Random Forest,-13.54,-19.05,-8.72,255
5,B_toan_bo_train → C_toan_bo_train,Random Forest,52.20,41.12,63.99,255
6,B_du_lich_su → C_lich_su,XGBoost,58.36,46.45,71.03,255
7,B_toan_bo_train → B_du_lich_su,XGBoost,-11.45,-15.86,-7.52,255
8,B_toan_bo_train → C_toan_bo_train,XGBoost,50.10,38.98,62.14,255
9,Seasonal naive → C_lich_su,XGBoost,134.05,101.89,170.98,255


**Nhận xét**

- Cùng tập train đủ lịch sử, thêm lịch sử giảm MAE 58,36 xe/giờ với XGBoost (CI 95%: 46,45–71,03), 62,17 với Random Forest và 47,09 với Ridge. Đây là mức cải thiện rõ nhất trong các phiên bản.
- Chỉ giữ giờ đủ lịch sử làm B kém đi 11,45 xe/giờ với XGBoost (CI: −15,86 đến −7,52). Vì vậy, khi so B–C cần kiểm soát số mẫu train như thiết kế trong README.
- So với B trên toàn bộ train, `C_toan_bo_train` giảm 50,10 xe/giờ với XGBoost (20,6%).
- Kết quả này giả định đã biết lưu lượng đến giờ `t − 1` (qua `mean_24h`) tại thời điểm dự đoán. Demo Time + Weather không dùng lịch sử nên không đạt mức sai số này.

## 4.4 · Ghi nhật ký thực nghiệm

In [7]:
for item in results:
    item["tap_danh_gia"] = f"validation đủ lịch sử; {len(validation_eligible)}/{len(validation)} giờ."

# Log chỉ được tạo khi chủ động bật; đặt biến môi trường SAVE_EXPERIMENT_LOGS=1
# hoặc đổi giá trị bên dưới thành True. Mỗi lần ghi tạo file mới, không ghi đè.
SAVE_EXPERIMENT_LOGS = os.environ.get("SAVE_EXPERIMENT_LOGS") == "1"

common_observations = run_observations(data_path, splits, "notebooks/BASELINE/baseline_V_4.ipynb")
common_problems = ['Đánh giá tuần tự từng giờ, không phải dự đoán nhiều bước.', 'Metrics chỉ tính trên các giờ đủ lịch sử; train bị giảm do thiếu giờ.']
next_steps = ['Phân tích sai số và SHAP; sau khi nhóm chốt quy trình mới đánh giá test trong V_5.']

if SAVE_EXPERIMENT_LOGS:
    for item in results:
        log_path = save_experiment_log(
            model=f"{item['config']} - {item['model']}",
            features=item["features"],
            results=item["metrics"],
            observations=common_observations + [
                "Tập đánh giá: " + item.get("tap_danh_gia", f"validation; {len(validation)} giờ."),
                f"Tham số được chọn: {item['search'].best_params_}",
                f"MAE trung bình CV: {-item['search'].best_score_}",
                f"Kết quả từng fold: {summarize_search(item['search']).to_json(orient='records', force_ascii=False)}",
            ],
            problems=common_problems,
            next_steps=next_steps,
        )
        print("Đã lưu:", log_path.name)
else:
    print("Chưa tạo log. Đặt SAVE_EXPERIMENT_LOGS=1 khi cần ghi.")

Đã lưu: EXP_019.txt


Đã lưu: EXP_020.txt
Đã lưu: EXP_021.txt
Đã lưu: EXP_022.txt
Đã lưu: EXP_023.txt
Đã lưu: EXP_024.txt
Đã lưu: EXP_025.txt
Đã lưu: EXP_026.txt
Đã lưu: EXP_027.txt


### Kết luận V_4

- Trên cùng 6.050 giờ: seasonal naive 330,18 → trung bình thứ–giờ 284,98 → B (XGBoost) 243,04 → C (XGBoost, toàn bộ train) **192,94** xe/giờ.
- Với XGBoost, lịch sử giảm MAE 50,10 xe/giờ, còn thời tiết giảm 1,49 xe/giờ (V_3, trên toàn validation).
- Chưa chạy V_5: cần nhóm chốt mô hình, quy tắc dữ liệu và kịch bản trước khi đánh giá test một lần.